In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
rho = 997.0  # kg/m^3, water at room temperature
time_window = None  # Example: (4475, 4525) or None for full dataset

# Sensor names you want to extract
UPSTREAM_NAME = "FM-PT"
DOWNSTREAM_NAME = "FI-PT"
FLOW_NAME = "FM-FM"

# ------------------------------------------------------------
# FUNCTION: Load and parse a single CSV
# ------------------------------------------------------------
def load_sensor_csv(path: Path):
    df = pd.read_csv(path, header=0)
    df["t_rel"] = df.t_wall - df.t_wall.min()
    return df

# ------------------------------------------------------------
# FUNCTION: Extract a single sensor's time/value arrays
# ------------------------------------------------------------
def extract_sensor(df, sensor_name, time_window=None):
    g = df[df["sensor_name"].str.strip() == sensor_name].copy()
    if g.empty:
        raise ValueError(f"Sensor '{sensor_name}' not found in file.")

    g = g.sort_values("t_rel")
    t = g.t_rel.to_numpy()
    v = g.value.to_numpy()

    # Apply time window if requested
    if time_window is not None:
        tmin, tmax = time_window
        mask = (t > tmin) & (t < tmax)
        t = t[mask]
        v = v[mask]

    return t, v

# ------------------------------------------------------------
# FUNCTION: Compute CdA
# ------------------------------------------------------------
def compute_cda(P_up, P_down, Q, rho):
    dP = P_up - P_down
    mdot = rho * Q
    CdA = mdot / np.sqrt(2 * rho * dP)
    return CdA

# ------------------------------------------------------------
# MAIN PIPELINE
# ------------------------------------------------------------
def process_file(path: Path):
    print(f"\n--- Processing {path.name} ---")
    df = load_sensor_csv(path)

    # Extract sensors
    t_up, P_up = extract_sensor(df, UPSTREAM_NAME, time_window)
    t_dn, P_dn = extract_sensor(df, DOWNSTREAM_NAME, time_window)
    t_fm, Q = extract_sensor(df, FLOW_NAME, time_window)

    # Time alignment check
    if not (len(t_up) == len(t_dn) == len(t_fm)):
        print("Time bases differ — interpolating onto common time grid.")
        t_common = np.intersect1d(np.intersect1d(t_up, t_dn), t_fm)
        P_up = np.interp(t_common, t_up, P_up)
        P_dn = np.interp(t_common, t_dn, P_dn)
        Q = np.interp(t_common, t_fm, Q)
        t = t_common
    else:
        t = t_up

    # Compute CdA
    CdA = compute_cda(P_up, P_dn, Q, rho)

    # Build output DataFrame
    out = pd.DataFrame({
        "time_s": t,
        "P_up": P_up,
        "P_down": P_dn,
        "Q_m3s": Q,
        "CdA": CdA
    })

    # Plot
    plt.figure(figsize=(10,5))
    plt.plot(t, CdA, label="CdA")
    plt.xlabel("Time (s)")
    plt.ylabel("CdA (m^2)")
    plt.title(f"CdA vs Time — {path.name}")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return out

# ------------------------------------------------------------
# RUN ON MULTIPLE FILES
# ------------------------------------------------------------
# Set your folder here:
data_folder = Path("C:\my_stuff\Capstone\data_analysis\04\18-17-09_data_csv")

csv_files = list(data_folder.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

all_results = []
for f in csv_files:
    try:
        result = process_file(f)
        all_results.append(result)
    except Exception as e:
        print(f"Error processing {f.name}: {e}")

# Combine results if desired
if all_results:
    combined = pd.concat(all_results, keys=[f.name for f in csv_files])
    print("\nCombined results available in variable 'combined'")
